In [13]:
# =====================================================
#  Data Cleaning - Rider dataset cleaning
# =====================================================

import pandas as pd
import numpy as np

In [14]:
# =====================================================
# Load Raw Datasets
# =====================================================

riders = pd.read_csv(
    "../data/raw/riders.csv",
    parse_dates=["signup_date"]
)

trips = pd.read_csv(
    "../data/raw/trips.csv",
    parse_dates=["pickup_time", "dropoff_time"]
)

drivers = pd.read_csv(
    "../data/raw/drivers.csv",
    parse_dates=["signup_date", "last_active"]
)

sessions = pd.read_csv(
    "../data/raw/sessions.csv",
    parse_dates=["session_time"]
)

promotions = pd.read_csv(
    "../data/raw/promotions.csv",
    parse_dates=["start_date", "end_date"]
)

In [15]:
# =====================================================
# Clean Riders Dataset
# =====================================================

riders_clean = riders.copy()

riders_clean = riders_clean.drop_duplicates()

text_cols = ["loyalty_status", "city", "referred_by"]

for col in text_cols:
    riders_clean[col] = (
        riders_clean[col]
        .astype("string")
        .str.strip()
        .str.title()
    )

riders_clean["referred_by"] = riders_clean["referred_by"].fillna("Unknown")

riders_clean["is_referred"] = (
    riders_clean["referred_by"] != "Unknown"
).astype(int)

riders_clean = riders_clean[
    riders_clean["age"].between(18, 100)
]

riders_clean = riders_clean[
    riders_clean["avg_rating_given"].between(0, 5)
]

# Validate churn_prob range before removing it
riders_clean = riders_clean[
    riders_clean["churn_prob"].between(0, 1)
]


# Remove churn_prob to prevent data leakage
riders_clean = riders_clean.drop(columns=["churn_prob"])

print("Original shape:", riders.shape)
print("Cleaned shape:", riders_clean.shape)
print("Duplicate rows:", riders_clean.duplicated().sum())

print("\nMissing values:")
print(riders_clean.isnull().sum())

display(riders_clean.head())

Original shape: (10000, 8)
Cleaned shape: (10000, 8)
Duplicate rows: 0

Missing values:
user_id             0
signup_date         0
loyalty_status      0
age                 0
city                0
avg_rating_given    0
referred_by         0
is_referred         0
dtype: int64


,user_id,signup_date,loyalty_status,age,city,avg_rating_given,referred_by,is_referred
0,R00000,2025-01-24,Bronze,34.729629,Nairobi,5.0,R00001,1
1,R00001,2024-09-09,Bronze,34.571020,Nairobi,4.7,Unknown,0
2,R00002,2024-09-07,Bronze,47.133960,Lagos,4.2,Unknown,0
3,R00003,2025-03-17,Bronze,41.658628,Nairobi,4.9,Unknown,0
4,R00004,2024-08-20,Silver,40.681709,Lagos,3.9,R00002,1


In [16]:
riders_clean.to_csv(
    "../data/processed/riders_clean.csv",
    index=False
)

print("Riders dataset cleaned and saved successfully.")

Riders dataset cleaned and saved successfully.


In [17]:
# =====================================================
# Clean Trips Dataset
# =====================================================

trips_clean = trips.copy()

trips_clean = trips_clean.drop_duplicates()
trips_clean = trips_clean.drop_duplicates(subset=["trip_id"], keep="first")

trips_clean["pickup_time"] = pd.to_datetime(
    trips_clean["pickup_time"],
    errors="coerce",
    utc=True
)

trips_clean["dropoff_time"] = pd.to_datetime(
    trips_clean["dropoff_time"],
    errors="coerce",
    utc=True
)

trips_clean = trips_clean.dropna(subset=[
    "trip_id",
    "user_id",
    "driver_id",
    "pickup_time",
    "dropoff_time",
    "fare",
    "tip",
    "surge_multiplier"
])

trips_clean = trips_clean[
    trips_clean["dropoff_time"] > trips_clean["pickup_time"]
]

trips_clean["trip_duration_minutes"] = (
    trips_clean["dropoff_time"] - trips_clean["pickup_time"]
).dt.total_seconds() / 60

trips_clean = trips_clean[trips_clean["fare"] >= 0]
trips_clean = trips_clean[trips_clean["tip"] >= 0]
trips_clean = trips_clean[trips_clean["surge_multiplier"] >= 1]

trips_clean = trips_clean[
    trips_clean["trip_duration_minutes"].between(1, 300)
]

text_cols = [
    "payment_type",
    "weather",
    "city",
    "loyalty_status"
]

for col in text_cols:
    trips_clean[col] = (
        trips_clean[col]
        .astype("string")
        .str.strip()
        .str.title()
    )

print("Original shape:", trips.shape)
print("Cleaned shape:", trips_clean.shape)
print("Duplicate rows:", trips_clean.duplicated().sum())
print("Duplicate Trip IDs:", trips_clean["trip_id"].duplicated().sum())

print("\nTrip duration summary:")
print(trips_clean["trip_duration_minutes"].describe())

print("\nMissing values:")
print(trips_clean.isnull().sum())

display(trips_clean.head())

Original shape: (200000, 16)
Cleaned shape: (200000, 17)
Duplicate rows: 0
Duplicate Trip IDs: 0

Trip duration summary:
count    200000.000000
mean         31.957310
std          15.871754
min           5.000000
25%          18.000000
50%          32.000000
75%          46.000000
max          59.000000
Name: trip_duration_minutes, dtype: float64

Missing values:
trip_id                  0
user_id                  0
driver_id                0
fare                     0
surge_multiplier         0
tip                      0
payment_type             0
pickup_time              0
dropoff_time             0
pickup_lat               0
pickup_lng               0
dropoff_lat              0
dropoff_lng              0
weather                  0
city                     0
loyalty_status           0
trip_duration_minutes    0
dtype: int64


,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status,trip_duration_minutes
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 16:14:50+00:00,2024-11-27 17:06:50+00:00,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze,52.0
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 22:59:48+00:00,2024-10-28 23:12:48+00:00,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold,13.0
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 03:09:41+00:00,2025-02-17 03:25:41+00:00,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze,16.0
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 17:22:14+00:00,2024-06-18 17:27:14+00:00,29.819554,31.188780,29.837689,31.232978,Cloudy,Cairo,Bronze,5.0
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 07:31:16+00:00,2024-10-05 08:01:16+00:00,-1.676479,36.729219,-1.638395,36.694063,Sunny,Nairobi,Gold,30.0


In [18]:
trips_clean.to_csv(
    "../data/processed/trips_clean.csv",
    index=False
)

print("Trips dataset cleaned and saved successfully.")

Trips dataset cleaned and saved successfully.


In [19]:
# =====================================================
# Clean Drivers Dataset
# =====================================================

drivers_clean = drivers.copy()

drivers_clean = drivers_clean.drop_duplicates()
drivers_clean = drivers_clean.drop_duplicates(subset=["driver_id"], keep="first")

drivers_clean["signup_date"] = pd.to_datetime(
    drivers_clean["signup_date"],
    errors="coerce"
)

drivers_clean["last_active"] = pd.to_datetime(
    drivers_clean["last_active"],
    errors="coerce"
)

drivers_clean = drivers_clean.dropna(subset=[
    "driver_id",
    "rating",
    "vehicle_type",
    "signup_date",
    "last_active",
    "city",
    "acceptance_rate"
])

drivers_clean = drivers_clean[
    drivers_clean["rating"].between(1, 5)
]

drivers_clean = drivers_clean[
    drivers_clean["acceptance_rate"].between(0, 1)
]

drivers_clean = drivers_clean[
    drivers_clean["last_active"] >= drivers_clean["signup_date"]
]

text_cols = ["vehicle_type", "city"]

for col in text_cols:
    drivers_clean[col] = (
        drivers_clean[col]
        .astype("string")
        .str.strip()
        .str.title()
    )

print("Original shape:", drivers.shape)
print("Cleaned shape:", drivers_clean.shape)
print("Duplicate rows:", drivers_clean.duplicated().sum())
print("Duplicate Driver IDs:", drivers_clean["driver_id"].duplicated().sum())

print("\nMissing values:")
print(drivers_clean.isnull().sum())

display(drivers_clean.head())

Original shape: (5000, 7)
Cleaned shape: (4772, 7)
Duplicate rows: 0
Duplicate Driver IDs: 0

Missing values:
driver_id          0
rating             0
vehicle_type       0
signup_date        0
last_active        0
city               0
acceptance_rate    0
dtype: int64


,driver_id,rating,vehicle_type,signup_date,last_active,city,acceptance_rate
1,D00001,5.0,Sedan,2023-03-27,2025-04-27 01:44:02.472554,Nairobi,0.548786
2,D00002,4.5,Motorcycle,2024-05-02,2025-03-07 19:24:46.367672,Nairobi,0.593724
3,D00003,5.0,Motorcycle,2023-04-16,2025-03-26 19:16:24.253793,Nairobi,0.990000
4,D00004,4.4,Motorcycle,2023-05-28,2025-04-08 18:54:44.649615,Lagos,0.519773
5,D00005,3.1,Sedan,2024-09-27,2024-12-15 23:26:07.576316,Nairobi,0.874726


In [20]:
drivers_clean.to_csv(
    "../data/processed/drivers_clean.csv",
    index=False
)

print("Drivers dataset cleaned and saved successfully.")

Drivers dataset cleaned and saved successfully.


In [21]:
# =====================================================
# Clean Sessions Dataset
# =====================================================

sessions_clean = sessions.copy()

sessions_clean = sessions_clean.drop_duplicates()

sessions_clean["session_time"] = pd.to_datetime(
    sessions_clean["session_time"],
    errors="coerce",
    utc=True
)

sessions_clean = sessions_clean.dropna(subset=[
    "session_id",
    "rider_id",
    "session_time",
    "time_on_app",
    "pages_visited",
    "converted",
    "city",
    "loyalty_status"
])

sessions_clean = sessions_clean[
    sessions_clean["time_on_app"] >= 0
]

sessions_clean = sessions_clean[
    sessions_clean["pages_visited"].between(1, 5)
]

sessions_clean = sessions_clean[
    sessions_clean["converted"].isin([0, 1])
]

text_cols = ["city", "loyalty_status"]

for col in text_cols:
    sessions_clean[col] = (
        sessions_clean[col]
        .astype("string")
        .str.strip()
        .str.title()
    )

print("Original shape:", sessions.shape)
print("Cleaned shape:", sessions_clean.shape)
print("Duplicate rows:", sessions_clean.duplicated().sum())
print("Duplicate Session IDs:", sessions_clean["session_id"].duplicated().sum())

print("\nMissing values:")
print(sessions_clean.isnull().sum())

display(sessions_clean.head())

Original shape: (50000, 8)
Cleaned shape: (50000, 8)
Duplicate rows: 0
Duplicate Session IDs: 0

Missing values:
session_id        0
rider_id          0
session_time      0
time_on_app       0
pages_visited     0
converted         0
city              0
loyalty_status    0
dtype: int64


,session_id,rider_id,session_time,time_on_app,pages_visited,converted,city,loyalty_status
0,S000000,R08605,2025-04-27 16:52:06+00:00,79,4,1,Cairo,Bronze
1,S000001,R08823,2025-04-27 05:05:22+00:00,101,3,0,Nairobi,Silver
2,S000002,R05342,2025-04-27 21:12:25+00:00,12,1,0,Cairo,Bronze
3,S000003,R05057,2025-04-27 14:26:25+00:00,19,1,0,Lagos,Silver
4,S000004,R09614,2025-04-27 08:17:22+00:00,4,1,0,Lagos,Bronze


In [22]:
sessions_clean.to_csv(
    "../data/processed/sessions_clean.csv",
    index=False
)

print("Sessions dataset cleaned and saved successfully.")

Sessions dataset cleaned and saved successfully.


In [23]:
# =====================================================
# Clean Promotions Dataset
# =====================================================

promotions_clean = promotions.copy()

promotions_clean = promotions_clean.drop_duplicates()
promotions_clean = promotions_clean.drop_duplicates(subset=["promo_id"], keep="first")

promotions_clean["start_date"] = pd.to_datetime(
    promotions_clean["start_date"],
    errors="coerce"
)

promotions_clean["end_date"] = pd.to_datetime(
    promotions_clean["end_date"],
    errors="coerce"
)

promotions_clean = promotions_clean.dropna(subset=[
    "promo_id",
    "promo_name",
    "promo_type",
    "promo_value",
    "start_date",
    "end_date",
    "target_segment",
    "city_scope"
])

promotions_clean = promotions_clean[
    promotions_clean["end_date"] > promotions_clean["start_date"]
]

promotions_clean = promotions_clean[
    promotions_clean["promo_value"] > 0
]

promotions_clean["promo_duration_days"] = (
    promotions_clean["end_date"] - promotions_clean["start_date"]
).dt.days

text_cols = [
    "promo_name",
    "promo_type",
    "target_segment",
    "city_scope",
    "success_metric"
]

for col in text_cols:
    if col in promotions_clean.columns:
        promotions_clean[col] = (
            promotions_clean[col]
            .astype("string")
            .str.strip()
            .str.title()
        )

print("Original shape:", promotions.shape)
print("Cleaned shape:", promotions_clean.shape)
print("Duplicate rows:", promotions_clean.duplicated().sum())
print("Duplicate Promo IDs:", promotions_clean["promo_id"].duplicated().sum())

print("\nMissing values:")
print(promotions_clean.isnull().sum())

display(promotions_clean.head())

Original shape: (20, 11)
Cleaned shape: (20, 12)
Duplicate rows: 0
Duplicate Promo IDs: 0

Missing values:
promo_id               0
promo_name             0
promo_type             0
promo_value            0
start_date             0
end_date               0
target_segment         0
city_scope             0
ab_test_groups         0
test_allocation        0
success_metric         0
promo_duration_days    0
dtype: int64


,promo_id,promo_name,promo_type,promo_value,start_date,end_date,target_segment,city_scope,ab_test_groups,test_allocation,success_metric,promo_duration_days
0,P000,Peak Hour Pass,Surge_Waiver,1.0,2025-04-26,2025-05-25,All,Nairobi,['All'],[1.0],Usage Frequency,29
1,P001,Peak Hour Pass,Surge_Waiver,1.0,2025-04-26,2025-05-22,All,Cairo,"['Control', 'Variant A', 'Variant B']","[0.3, 0.4, 0.3]",Conversion Rate,26
2,P002,Peak Hour Pass,Surge_Waiver,1.0,2025-04-26,2025-05-16,All,Cairo,"['Control', 'Variant A', 'Variant B']","[0.3, 0.4, 0.3]",Roi,20
3,P003,Loyalty Bonus,Points,100.0,2025-04-26,2025-05-04,Gold+,Nairobi,"['Control', 'Variant A', 'Variant B']","[0.3, 0.4, 0.3]",Conversion Rate,8
4,P004,Loyalty Bonus,Points,100.0,2025-04-26,2025-05-15,Gold+,Nairobi,['All'],[1.0],Usage Frequency,19


In [24]:
promotions_clean.to_csv(
    "../data/processed/promotions_clean.csv",
    index=False
)

print("Promotions dataset cleaned and saved successfully.")

Promotions dataset cleaned and saved successfully.
